# Fine-tuning Transformers for Kinyarwanda Sentiment (AfriSenti-Kin)

**Runtime → Change runtime type → T4 GPU** before running.

This notebook runs `src/finetune_transformer.py`, which fine-tunes mBERT, AfriBERTa-base and AfroXLMR-base (3 seeds each, ~25–35 min on a T4), then re-runs the error analysis so the experiment figure includes the Transformers.

In [ ]:
# 1. Get the code
!git clone https://github.com/belladev250/kin-sentiment.git
%cd kin-sentiment
!pip -q install -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Classical baselines + BiLSTM (optional here — results are already in `results/`)
Uncomment to reproduce them on Colab.

In [ ]:
# !python src/baselines.py
# !python src/bilstm.py
# !python src/annotator_agreement.py

## 3. Fine-tune the Transformers
Add `--large` to also try AfroXLMR-large (slower, ~15 extra min). Short on time? Use `--only T3 --seeds 1` (≈4 min).

In [ ]:
!python src/finetune_transformer.py

## 4. Error analysis + figures (now including Transformer bars)

In [ ]:
!python src/error_analysis.py
!python src/transformer_analysis.py
!python src/significance.py
import pandas as pd
pd.read_csv('results/transformer_experiments.csv')[['id','experiment','dev_macro_f1','test_macro_f1','test_macro_f1_std','test_weighted_f1']].round(4)

In [ ]:
from IPython.display import Image
Image('results/figures/experiments.png')

## 5. Upload the best model to the Hugging Face Hub (for the web app)
Create a free account at huggingface.co → Settings → Access Tokens → new token with **write** access.

In [ ]:
from huggingface_hub import login
login()   # paste a Hugging Face token with WRITE access into the box

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
REPO = 'bellahug3/kin-sentiment-transformer'
AutoModelForSequenceClassification.from_pretrained('models/best_transformer').push_to_hub(REPO)
AutoTokenizer.from_pretrained('models/best_transformer').push_to_hub(REPO)
print('https://huggingface.co/' + REPO)

## 6. Download the results to commit them to GitHub

In [ ]:
!zip -qr transformer_results.zip results
from google.colab import files
files.download('transformer_results.zip')